## Imports

In [ ]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

import neuro_fuzzy_toolbox as nft

In [3]:
import numpy as np

In [4]:
from sklearn.preprocessing import MinMaxScaler

# Surface (1k)

## Data

In [5]:
def z(x, y):
  return ((3) * ((1-x)**2) * (np.exp(-(x**2)-((y+1)**2))) - (10) * ((x/5)-(x**3)-(y**5)) * (np.exp(-(x**2)-(y**2))) - (1/3)*np.exp(-(x+1)**2-(y**2)))

#Training
x0 = np.random.uniform(-3,3,1000)
x1 = np.random.uniform(-3,3,1000)

e = np.random.normal(0,0.5,1000) #noise
Y = z(x0,x1) + e

#Testing
x0_test = np.random.uniform(-3,3,1000)
x1_test = np.random.uniform(-3,3,1000)

Y_test = z(x0_test,x1_test)

In [6]:
#Training
scaler = MinMaxScaler(feature_range=(0, 1))
vstack_train = np.vstack((x0,x1)).T
scaled_train = scaler.fit_transform(vstack_train)

#Testing
vstack_test = np.vstack((x0_test,x1_test)).T
scaled_test = scaler.transform(vstack_test)

In [7]:
dtype = torch.float64

train_loader = data.DataLoader(
    data.TensorDataset(
        torch.tensor(scaled_train, dtype=dtype), 
        torch.tensor(Y, dtype=dtype)), 
    batch_size = 8,
    shuffle = True)

x_train = train_loader.dataset.tensors[0]
y_train = train_loader.dataset.tensors[1]

x_test = torch.tensor(scaled_test, dtype=dtype)
y_test = torch.tensor(Y_test, dtype=dtype)

In [8]:
x_train.dtype

torch.float64

## Model & Training

### ANFIS

In [74]:
model = nft.rule_reduced_ANFIS(
    input_size = 2,
    num_mfs = 1,
    outputs = 1,
    dtype=torch.float64
)

In [75]:
model.init_premises(x_train)

### Hybrid Learning Algorithm

In [76]:
loss_fn = nn.functional.mse_loss
#loss_fn = nn.functional.binary_cross_entropy
#loss_fn = nn.functional.cross_entropy

optimizer = torch.optim.AdamW
params = {'lr': 0.0001, 'weight_decay': 0.001}

early_stopping = nft.EarlyStopping(patience=40, delta=0.001)

In [77]:
trainer = nft.Hybrid_learning_algorithm(
    epochs=500,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    early_stopping=early_stopping
)

### SONFIS

In [78]:
Ngrow = 30
dGrow = 0.9
Nsplit = 30
eSplit = 0.6
Nvanish = 10
lVanish = 3

max_iterations = 40

anfis_trainer = trainer

validation = 0.3
sonfis_early_stopping = nft.EarlyStopping(patience=10)
last_training_iteration = True

In [79]:
dGrow**x_train.shape[1]

0.81

In [80]:
sonfis = nft.SONFIS(
    Ngrow=Ngrow,
    dGrow=dGrow,
    Nsplit=Nsplit,
    eSplit=eSplit,
    Nvanish=Nvanish,
    lVanish=lVanish,
    max_iterations=max_iterations,
    ANFIStrainer=anfis_trainer,
    validation=validation,
    early_stopping=sonfis_early_stopping,
    last_training_iteration=last_training_iteration
)

In [81]:
%time sonfis(model, train_loader, verbose=True)

Iteration:  0/40 - loss: 4.349988 - validation loss: 4.804977
 -> ANFIS rules: 1

Iteration:  1/40 - loss: 3.701048 - validation loss: 4.010757
 -> ANFIS rules: 2

Iteration:  2/40 - loss: 6.218348 - validation loss: 6.772489
 -> ANFIS rules: 3

Iteration:  3/40 - loss: 4.072347 - validation loss: 4.216559
 -> ANFIS rules: 5

Iteration:  4/40 - loss: 3.755841 - validation loss: 3.991636
 -> ANFIS rules: 6

Iteration:  5/40 - loss: 5.033733 - validation loss: 5.275037
 -> ANFIS rules: 8

Iteration:  6/40 - loss: 1.845770 - validation loss: 2.029174
 -> ANFIS rules: 14

Iteration:  7/40 - loss: 1.546570 - validation loss: 1.785334
 -> ANFIS rules: 16

Iteration:  8/40 - loss: 1.477493 - validation loss: 1.711044
 -> ANFIS rules: 17

Iteration:  9/40 - loss: 2.616519 - validation loss: 2.844619
 -> ANFIS rules: 23

Iteration: 10/40 - loss: 2.499915 - validation loss: 2.793168
 -> ANFIS rules: 25

Iteration: 11/40 - loss: 4.279316 - validation loss: 5.634010
 -> ANFIS rules: 29

Iteration:

In [82]:
test_measures = nft.get_measures(model, x_test, y_test)

for measure in test_measures:
    print(measure + ':', test_measures[measure])

MSE: 0.15166134107489626
RMSE: 0.38943721069627674
MAE: 0.2996449422370423
R2: 0.9575824522750257
MAPE: 78.5236074914712


In [83]:
train_measures = nft.get_measures(model, x_train, y_train)

for measure in train_measures:
    print(measure + ':', train_measures[measure])

MSE: 0.25769565677178896
RMSE: 0.5076373279929176
MAE: 0.4000552397178467
R2: 0.9262385510521839
MAPE: 1.9458225662895203
